# Notebook: 01_rt01_exploracao_inicial.ipynb
**id_rastreabilidade:** RTB_001  
**Versão:** v01  
**Data de criação:** "24/03/2026"  
**Última atualização:** "24/03/2026"  
**Responsável:** Ailton Vendramini

---

## Finalidade
Explorar a base CadÚnico `tudo.csv`, validar as colunas críticas do dicionário operacional e calcular preliminarmente a variável RT_01 do IVS-H (% de famílias com renda per capita menor ou igual a 1/2 salário mínimo).

---

## Base Conceitual
Documentos GitHub que definem o quê e o porquê:
- `dim_variavel_IVS_v01r7.md` → RT_01; Nota Metodológica; Resumo de Disponibilidade
- `conceito_vulnerabilidade_v03.md` → limites de cobertura do CadÚnico e uso analítico da vulnerabilidade
- `arquitetura_dados_IVS_IBGE_Horto_v10.md` → camada de dados e lógica geral do Atlas

> **⚠️ Fronteira IVS-H × IPST-H:** este notebook calcula variáveis do
> IVS-H — exatamente as 16 variáveis do IVS/IPEA (IU_01–IU_03, CH_01–CH_08,
> RT_01–RT_05). Variáveis de deslocamento pertencem ao IPST-H e não devem
> ser calculadas, armazenadas ou inseridas em `fato_variavel_ivs_loteamento`.
> Referência: `dim_variavel_IVS_v01r7.md` — Nota Metodológica.

## Base Lógica
Artefatos que definem a estrutura de dados:
- `schema_IVS_v05.sql` → leitura exploratória; futuras tabelas de staging e fato
- `nota_tecnica_fato_ivs_loteamento_v04.sql` → separação entre fato de variáveis e fato do índice
- `matriz_rastreabilidade_operacional_v02.md` → RTB_001

---

## Fontes de Dados
**Arquivo de entrada:** `/tmp/cecad/tudo.csv`  
**Versão do dado:** `cadunico_v7_2025_12`  
**Periodo de referência:** `CadÚnico dez/2025`  
**Dicionário utilizado:** `dicionario_cadunico_operacional_v02.md`  
**Banco SQLite:** `hortolandia.db`  
**⚠️ Risco de schema:** Alto — verificar dicionário antes de nova carga

---

## Tabelas
**Leitura:** `STG_CADUNICO_RAW`  
**Escrita:** —  

---

## Outputs Gerados
| Caminho completo | tipo_output | Pode commitar? |
| --- | --- | --- |
| `outputs/tabelas/ivs_variaveis.csv` | exploratorio | Não |

---

## Variáveis IVS-H Calculadas
- [x] RT_01 — % famílias renda per capita menor ou igual a 1/2 SM  
- [ ] RT_04 — % domicílios dependentes de idosos  
- [ ] CH_05 — % mães chefes sem fund. completo, filho menor de 15  
- [ ] CH_06 — Taxa de analfabetismo 15 anos ou mais  
- [ ] CH_07 — % crianças sem morador com fund. completo

In [2]:
import pandas as pd
import glob

# Lista todos os arquivos
arquivos = glob.glob("2026_*.csv")

dfs = []

for a in arquivos:
    try:
        df = pd.read_csv(
            a,
            sep=",",
            encoding="utf-8",
            engine="python",
            on_bad_lines="skip"
        )

        # limpeza básica
        df.columns = [str(c).strip() for c in df.columns]

        # garante colunas mínimas
        if "dimensao_ivs" not in df.columns:
            print(f"IGNORADO (sem estrutura): {a}")
            continue

        dfs.append(df)

    except Exception as e:
        print(f"ERRO em {a}: {e}")

# consolidação
df_total = pd.concat(dfs, ignore_index=True, sort=False)

# remove evento político (Clodoaldo)
df_total = df_total[
    ~df_total["item"].str.contains("Clodoaldo", na=False)
]

# remove registros sem polaridade
df_total = df_total[
    df_total["polaridade_evento"].notna()
]

# =========================
# RESULTADOS FINAIS
# =========================

print("=== BASE FINAL ===")
print("Total de eventos:", len(df_total))

print("\nPolaridade:")
print(df_total["polaridade_evento"].value_counts())

print("\nDimensões:")
print(df_total["dimensao_ivs"].value_counts())

ValueError: No objects to concatenate

In [ ]:
import os

pasta = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\series_jornalisticas'

conteudo = """fonte,data,item,pagina,dimensao_ivs,codigo_variavel,tipo_relacao_variavel,resumo,localidade,tipo_evento,gravidade,observacao,cod_loteamento,nivel_confianca_loteamento,polaridade_evento,tipo_origem_evento
Tribuna Liberal,2026-04-03,Monte Sinai — 152 UH e saneamento 650 familias,1,infraestrutura_urbana,IU_01,direta,152 unidades habitacionais para moradores de areas de risco em parceria com CDHU; entrega prevista 2o semestre 2026; redes subterraneas de esgoto e aguas pluviais para 650 familias ja concluidas em parceria com Sabesp,Monte Sinai,infraestrutura,alta,Evento de infraestrutura mais substantivo registrado no corpus; 650 familias com saneamento regularizado comparavel com IU_01; 152 UH insumo para estimar cobertura de familias em area de risco — cruzamento futuro com CadUnico,nao_identificado,medio,positivo,infraestrutura
Tribuna Liberal,2026-04-03,Vereador Clodoaldo — pre-candidatura deputado estadual pauta empregabilidade,7,renda_trabalho,SMIDS_GOV,contextual,Partido Podemos confirma pre-candidatura de Clodoaldo Santos a deputado estadual 2026; trajetoria pautada em empregabilidade e inclusao social; serie historica votos: 1016 (2016) — 2481 (2020) — 2916 (2024),Hortolandia,institucional,media,Sinaliza que pauta de empregabilidade e inclusao de Hortolandia tem tracao eleitoral em 2026; cria janela de oportunidade institucional para o Atlas,nao_aplicavel,nao_aplicavel,positivo,institucional
"""

caminho = os.path.join(pasta, '2026_04_03_tribuna_liberal.csv')
with open(caminho, 'w', encoding='utf-8') as f:
    f.write(conteudo)

print(f'✅ Arquivo criado: {caminho}')

In [3]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Correção dos CSVs Diários — Atlas Social de Hortolândia\n",
    "\n",
    "Este notebook corrige três problemas em todos os arquivos diários da série jornalística:\n",
    "\n",
    "1. **`localidade`** — qualquer valor fora de `Hortolândia` ou `Regional` vira `Hortolândia`\n",
    "2. **`cod_loteamento`** — decimais (`351.0` → `351`) e células vazias → `nao_aplicavel`\n",
    "3. **`nivel_confianca_loteamento`** — células vazias → `nao_aplicavel`"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import glob\n",
    "import os\n",
    "\n",
    "# Pasta dos arquivos diários\n",
    "pasta = r'C:\\Users\\ailto\\Atlas-Social-de-Hortolandia\\dados\\bd_externos\\series_jornalisticas'\n",
    "\n",
    "arquivos = sorted(glob.glob(os.path.join(pasta, '????_??_??_tribuna_liberal.csv')))\n",
    "\n",
    "print(f'Arquivos encontrados: {len(arquivos)}')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "localidades_validas = ['Hortolândia', 'Regional']\n",
    "total_corrigidos = 0\n",
    "\n",
    "for caminho in arquivos:\n",
    "    nome = os.path.basename(caminho)\n",
    "    df = pd.read_csv(caminho)\n",
    "    modificado = False\n",
    "    problemas = []\n",
    "\n",
    "    # 1. Localidade inválida\n",
    "    col_loc = 'localidade'\n",
    "    if col_loc in df.columns:\n",
    "        mask_loc = ~df[col_loc].isin(localidades_validas)\n",
    "        if mask_loc.any():\n",
    "            valores = df.loc[mask_loc, col_loc].unique().tolist()\n",
    "            problemas.append(f'localidade inválida: {valores}')\n",
    "            df.loc[mask_loc, col_loc] = 'Hortolândia'\n",
    "            modificado = True\n",
    "\n",
    "    # 2. cod_loteamento com decimal ou vazio\n",
    "    col_cod = 'cod_loteamento'\n",
    "    if col_cod in df.columns:\n",
    "        def corrigir_cod(val):\n",
    "            v = str(val).strip()\n",
    "            if v == '' or v == 'nan':\n",
    "                return 'nao_aplicavel'\n",
    "            try:\n",
    "                f = float(v)\n",
    "                if f == int(f):\n",
    "                    return str(int(f))\n",
    "                return v\n",
    "            except:\n",
    "                return v\n",
    "\n",
    "        novos = df[col_cod].apply(corrigir_cod)\n",
    "        if not novos.equals(df[col_cod].astype(str)):\n",
    "            problemas.append('cod_loteamento com decimal ou vazio')\n",
    "            df[col_cod] = novos\n",
    "            modificado = True\n",
    "\n",
    "    # 3. nivel_confianca_loteamento vazio\n",
    "    col_conf = 'nivel_confianca_loteamento'\n",
    "    if col_conf in df.columns:\n",
    "        mask_conf = df[col_conf].isna() | (df[col_conf].astype(str).str.strip() == '')\n",
    "        if mask_conf.any():\n",
    "            problemas.append(f'nivel_confianca vazio: {mask_conf.sum()} linha(s)')\n",
    "            df.loc[mask_conf, col_conf] = 'nao_aplicavel'\n",
    "            modificado = True\n",
    "\n",
    "    # Salva se houve modificação\n",
    "    if modificado:\n",
    "        df.to_csv(caminho, index=False)\n",
    "        total_corrigidos += 1\n",
    "        print(f'[CORRIGIDO] {nome}')\n",
    "        for p in problemas:\n",
    "            print(f'  → {p}')\n",
    "    else:\n",
    "        print(f'[OK] {nome}')\n",
    "\n",
    "print('=' * 60)\n",
    "print(f'Total corrigidos: {total_corrigidos}')\n",
    "print(f'Já estavam ok   : {len(arquivos) - total_corrigidos}')\n",
    "print('Concluído.')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}


NameError: name 'null' is not defined

In [1]:
import urllib.request
import os

pasta = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'

base_url = 'https://raw.githubusercontent.com/ailtonfv/Atlas-Social-de-Hortolandia/refs/heads/main/00_governan%C3%A7a/series_jornalisticas/'

arquivos = [
    '2026_03_14_tribuna_liberal.csv',
    '2026_03_18_tribuna_liberal.csv',
    '2026_03_21_tribuna_liberal.csv',
    '2026_03_24_tribuna_liberal.csv',
    '2026_03_25_tribuna_liberal.csv',
    '2026_03_26_tribuna_liberal.csv',
    '2026_03_27_tribuna_liberal.csv',
    '2026_03_28_tribuna_liberal.csv',
    '2026_03_29_tribuna_liberal.csv',
    '2026_03_31_tribuna_liberal.csv',
    '2026_04_01_tribuna_liberal.csv',
    '2026_04_02_tribuna_liberal.csv',
    'corpus_resumo_periodos.csv',
]

for arq in arquivos:
    destino = os.path.join(pasta, arq)
    url = base_url + arq
    urllib.request.urlretrieve(url, destino)
    print(f'✅ {arq}')

print('\nPronto.')

✅ 2026_03_14_tribuna_liberal.csv
✅ 2026_03_18_tribuna_liberal.csv
✅ 2026_03_21_tribuna_liberal.csv
✅ 2026_03_24_tribuna_liberal.csv
✅ 2026_03_25_tribuna_liberal.csv
✅ 2026_03_26_tribuna_liberal.csv
✅ 2026_03_27_tribuna_liberal.csv
✅ 2026_03_28_tribuna_liberal.csv
✅ 2026_03_29_tribuna_liberal.csv
✅ 2026_03_31_tribuna_liberal.csv
✅ 2026_04_01_tribuna_liberal.csv
✅ 2026_04_02_tribuna_liberal.csv
✅ corpus_resumo_periodos.csv

Pronto.


In [5]:
import pandas as pd

path = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral\corpus_resumo_periodos.csv'

df = pd.read_csv(path, sep=',')

df.loc[df['periodo'] == '2026-03', 'eventos_por_dimensao'] = 'CH:14 | RT:7 | IU:6 | MULTI:5 | GOV:2'

df.to_csv(path, sep=',', index=False)

print("Feito.")
print(df[['periodo','eventos_por_dimensao']])

Feito.
   periodo                   eventos_por_dimensao
0  2026-03  CH:14 | RT:7 | IU:6 | MULTI:5 | GOV:2


In [4]:
import pandas as pd

path = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral\corpus_resumo_periodos.csv'

df = pd.read_csv(path, sep=';')
print(df.columns.tolist())
print(df.head())

['periodo,total_eventos,eventos_negativos,eventos_positivos,eventos_por_loteamento,eventos_por_dimensao,observacao_sintese']
                                                                                                periodo,total_eventos,eventos_negativos,eventos_positivos,eventos_por_loteamento,eventos_por_dimensao,observacao_sintese
2026-03,34,26,8,Jd. Conceicao: 4 | Jd. Amanda: ... IU elevado reflete pressao urbana estrutural   primeiro registro institucional positivo via ...                                                                      


In [3]:
import pandas as pd

path = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral\corpus_resumo_periodos.csv'

df = pd.read_csv(path, sep=';')

df.loc[df['periodo'] == '2026-03', 'eventos_por_dimensao'] = 'CH:14 | RT:7 | IU:6 | MULTI:5 | GOV:2'

df.to_csv(path, sep=';', index=False)

print("Feito.")

KeyError: 'periodo'

In [2]:
import pandas as pd
import os

caminho = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'
nome_arquivo = '2026_04_02_tribuna_liberal.csv'
caminho_completo = os.path.join(caminho, nome_arquivo)

colunas = [
    'fonte', 'data', 'item', 'pagina', 'dimensao_ivs', 'codigo_variavel',
    'tipo_relacao_variavel', 'resumo', 'localidade', 'tipo_evento',
    'gravidade', 'observacao', 'cod_loteamento', 'nivel_confianca_loteamento',
    'polaridade_evento', 'tipo_origem_evento'
]

registros = [
    ['Tribuna Liberal', '2026-04-02',
     'Viaduto Vila Real entra na reta final em Hortolandia',
     12, 'infraestrutura_urbana', 'SMIDS_EXT', 'contextual',
     'Pavimentacao da Rua Argolino de Moraes concluida; rotatória e via a serem liberadas nos proximos dias; entrega total prevista entre final de abril e inicio de maio; prefeito Zeze Gomes acompanha obra',
     'Hortolandia', 'infraestrutura', 'media',
     'Infraestrutura viaria com potencial de reorganizacao do fluxo urbano e valorizacao territorial; impacto indireto futuro sobre IU_01 e IU_02; RP_6',
     63, 'medio', 'positivo', 'infraestrutura'],

    ['Tribuna Liberal', '2026-04-02',
     'GM registra 14 casos Lei Maria da Penha em marco — Hortolandia',
     4, 'capital_humano', 'CH_05', 'direta',
     'Balanco GM marco 2026; 14 ocorrencias Lei Maria da Penha registradas; 3316 ligacoes recebidas; 1068 com deslocamento de viatura; 304 BOs lavrados',
     'Hortolandia', 'violencia', 'alta',
     'Dado administrativo oficial auditavel; volume mensal LMP permite construcao de serie temporal; base para monitoramento padrao CH_05 por periodo',
     'nao_aplicavel', 'nao_aplicavel', 'negativo', 'seguranca'],

    ['Tribuna Liberal', '2026-04-02',
     'GM registra 36 averiguacoes criminais e ocorrencias multiplas em marco — Hortolandia',
     4, 'multidimensional', 'multidimensional', 'contextual',
     'Balanco GM marco 2026; 36 averiguacoes de fatos criminosos incluindo roubos furtos agressoes tentativas de feminicidio e homicidio; 480 atendimentos perturbacao sossego; 42 brigas; 68 apoios a orgaos publicos',
     'Hortolandia', 'violencia', 'alta',
     'Dado agregado mensal auditavel; volume e recorrencia permitem saida do evento isolado para padrao mensal; interface CH e RT; referencia para serie temporal do Atlas',
     'nao_aplicavel', 'nao_aplicavel', 'negativo', 'seguranca'],

    ['Tribuna Liberal', '2026-04-02',
     'Audiencia Publica Plano Diretor — Hortolandia 07/04/2026',
     4, 'governanca', 'SMIDS_GOV', 'contextual',
     'Audiencia publica para discussao do PLC 11/2025 — Plano Diretor do Municipio de Hortolandia; realizada na Camara Municipal em 07/04 as 19h; transmissao ao vivo YouTube',
     'Hortolandia', 'institucional', 'media',
     'Instrumento estruturante que define a base territorial do Atlas — RPs e loteamentos; legitima politicamente o modelo; acompanhar desdobramentos para ajustes em dim_loteamento',
     'nao_aplicavel', 'nao_aplicavel', 'positivo', 'institucional'],
]

df_novo = pd.DataFrame(registros, columns=colunas)
df_novo.to_csv(caminho_completo, index=False, encoding='utf-8-sig', sep=',')

print(f"Arquivo salvo: {caminho_completo}")
print(f"Registros: {len(df_novo)}")
df_novo

Arquivo salvo: C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral\2026_04_02_tribuna_liberal.csv
Registros: 4


,fonte,data,item,pagina,dimensao_ivs,codigo_variavel,tipo_relacao_variavel,resumo,localidade,tipo_evento,gravidade,observacao,cod_loteamento,nivel_confianca_loteamento,polaridade_evento,tipo_origem_evento
0,Tribuna Liberal,2026-04-02,Viaduto Vila Real entra na reta final em Horto...,12,infraestrutura_urbana,SMIDS_EXT,contextual,Pavimentacao da Rua Argolino de Moraes conclui...,Hortolandia,infraestrutura,media,Infraestrutura viaria com potencial de reorgan...,63,medio,positivo,infraestrutura
1,Tribuna Liberal,2026-04-02,GM registra 14 casos Lei Maria da Penha em mar...,4,capital_humano,CH_05,direta,Balanco GM marco 2026; 14 ocorrencias Lei Mari...,Hortolandia,violencia,alta,Dado administrativo oficial auditavel; volume ...,nao_aplicavel,nao_aplicavel,negativo,seguranca
2,Tribuna Liberal,2026-04-02,GM registra 36 averiguacoes criminais e ocorre...,4,multidimensional,multidimensional,contextual,Balanco GM marco 2026; 36 averiguacoes de fato...,Hortolandia,violencia,alta,Dado agregado mensal auditavel; volume e recor...,nao_aplicavel,nao_aplicavel,negativo,seguranca
3,Tribuna Liberal,2026-04-02,Audiencia Publica Plano Diretor — Hortolandia ...,4,governanca,SMIDS_GOV,contextual,Audiencia publica para discussao do PLC 11/202...,Hortolandia,institucional,media,Instrumento estruturante que define a base ter...,nao_aplicavel,nao_aplicavel,positivo,institucional


In [1]:
import pandas as pd
import os

caminho = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'
nome_arquivo = 'corpus_resumo_periodos.csv'
caminho_completo = os.path.join(caminho, nome_arquivo)

colunas = [
    'periodo',
    'total_eventos',
    'eventos_negativos',
    'eventos_positivos',
    'eventos_por_loteamento',
    'eventos_por_dimensao',
    'observacao_sintese'
]

# Primeiro registro — março 2026
registros = [
    [
        '2026-03',
        34,
        26,
        8,
        'Jd. Conceicao: 4 | Jd. Amanda: 3 | Parque Santo Andre: 2',
        'CH:14 | RT:7 | IU:6 | MULTI:5 | GOV:1 | EXT:1',
        'Concentracao de eventos CH no Jd. Conceicao com presenca simultanea de violencia domestica e vulnerabilidade juvenil; IU elevado reflete pressao urbana estrutural; primeiro registro institucional positivo via Selo FNAS sinaliza capacidade de resposta da rede'
    ]
]

df_resumo = pd.DataFrame(registros, columns=colunas)
df_resumo.to_csv(caminho_completo, index=False, encoding='utf-8-sig', sep=',')

print(f"Arquivo salvo: {caminho_completo}")
df_resumo

Arquivo salvo: C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral\corpus_resumo_periodos.csv


,periodo,total_eventos,eventos_negativos,eventos_positivos,eventos_por_loteamento,eventos_por_dimensao,observacao_sintese
0,2026-03,34,26,8,Jd. Conceicao: 4 | Jd. Amanda: 3 | Parque Sant...,CH:14 | RT:7 | IU:6 | MULTI:5 | GOV:1 | EXT:1,Concentracao de eventos CH no Jd. Conceicao co...


In [2]:
import pandas as pd
import os
import glob

caminho = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'
arquivos = sorted(glob.glob(os.path.join(caminho, '*.csv')))
print(f"Arquivos encontrados: {len(arquivos)}")

# Mapeamento tipo_origem_evento a partir de tipo_evento
mapa_origem = {
    'violencia'      : 'seguranca',
    'vulnerabilidade': 'assistencial',
    'politica_publica': 'institucional',
    'infraestrutura' : 'infraestrutura',
    'institucional'  : 'institucional',
    'problema'       : 'seguranca',  # default conservador — revisar caso a caso
}

for arquivo in arquivos:
    df = pd.read_csv(arquivo, sep=None, engine='python', encoding='utf-8-sig')
    alteracoes = []

    # 1. Adicionar tipo_origem_evento se ausente
    if 'tipo_origem_evento' not in df.columns:
        df['tipo_origem_evento'] = df['tipo_evento'].map(mapa_origem).fillna('a_revisar')
        alteracoes.append('tipo_origem_evento adicionado')

    # 2. SMIDS_EXT só onde não há mapeamento defensável para variável IVS
    # Linha 3 dos arquivos: violência de gênero → CH_05
    mask_genero = (
        df['codigo_variavel'].astype(str).str.strip() == 'SMIDS_EXT'
    ) & (
        df['resumo'].astype(str).str.contains('estupro|violencia de genero|agress', case=False, na=False)
    )
    if mask_genero.any():
        df.loc[mask_genero, 'codigo_variavel'] = 'CH_05'
        alteracoes.append('CH_05 corrigido em violência de gênero')

    # 3. a_revisar → contextual + governanca (emenda/CRAS/investimento)
    mask_revisar = (
        df['tipo_relacao_variavel'].astype(str).str.strip() == 'a_revisar'
    )
    if mask_revisar.any():
        df.loc[mask_revisar, 'tipo_relacao_variavel'] = 'contextual'
        df.loc[mask_revisar, 'codigo_variavel'] = 'governanca'
        alteracoes.append('a_revisar → contextual/governanca')

    df.to_csv(arquivo, index=False, encoding='utf-8-sig', sep=',')
    print(f"✓ {os.path.basename(arquivo)} — {', '.join(alteracoes) if alteracoes else 'sem alterações'}")

print("\nMigração concluída.")
print("⚠️  Revisar registros tipo_evento=problema com tipo_origem_evento=seguranca — inferência conservadora.")

Arquivos encontrados: 11
✓ 2026_03_14_tribuna_liberal.csv — tipo_origem_evento adicionado
✓ 2026_03_18_tribuna_liberal.csv — tipo_origem_evento adicionado
✓ 2026_03_21_tribuna_liberal.csv — tipo_origem_evento adicionado
✓ 2026_03_24_tribuna_liberal.csv — tipo_origem_evento adicionado
✓ 2026_03_25_tribuna_liberal.csv — tipo_origem_evento adicionado
✓ 2026_03_26_tribuna_liberal.csv — tipo_origem_evento adicionado
✓ 2026_03_27_tribuna_liberal.csv — tipo_origem_evento adicionado
✓ 2026_03_28_tribuna_liberal.csv — tipo_origem_evento adicionado, CH_05 corrigido em violência de gênero, a_revisar → contextual/governanca
✓ 2026_03_29_tribuna_liberal.csv — tipo_origem_evento adicionado
✓ 2026_03_31_tribuna_liberal.csv — tipo_origem_evento adicionado
✓ 2026_04_01_tribuna_liberal.csv — tipo_origem_evento adicionado

Migração concluída.
⚠️  Revisar registros tipo_evento=problema com tipo_origem_evento=seguranca — inferência conservadora.


In [4]:
import pandas as pd
import os
import glob

caminho = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'
arquivos = sorted(glob.glob(os.path.join(caminho, '*.csv')))

# Palavras-chave que indicam origem econômica
keywords_economico = [
    'gasolina', 'combustivel', 'preco', 'inflacao', 'custo de vida',
    'reajuste', 'tarifa', 'salario', 'renda', 'aluguel', 'energia'
]

for arquivo in arquivos:
    df = pd.read_csv(arquivo, sep=None, engine='python', encoding='utf-8-sig')
    
    if 'tipo_origem_evento' not in df.columns:
        continue

    mask_economico = (
        df['tipo_origem_evento'] == 'seguranca'
    ) & (
        df['resumo'].astype(str).str.contains(
            '|'.join(keywords_economico), case=False, na=False
        )
    )

    corrigidos = mask_economico.sum()
    if corrigidos > 0:
        df.loc[mask_economico, 'tipo_origem_evento'] = 'economico'
        df.to_csv(arquivo, index=False, encoding='utf-8-sig', sep=',')
        print(f"✓ {os.path.basename(arquivo)} — {corrigidos} registro(s) → economico")
    else:
        print(f"  {os.path.basename(arquivo)} — sem alterações")

print("\nCorreção concluída.")

  2026_03_14_tribuna_liberal.csv — sem alterações
  2026_03_18_tribuna_liberal.csv — sem alterações
  2026_03_21_tribuna_liberal.csv — sem alterações
  2026_03_24_tribuna_liberal.csv — sem alterações
✓ 2026_03_25_tribuna_liberal.csv — 1 registro(s) → economico
  2026_03_26_tribuna_liberal.csv — sem alterações
  2026_03_27_tribuna_liberal.csv — sem alterações
  2026_03_28_tribuna_liberal.csv — sem alterações
  2026_03_29_tribuna_liberal.csv — sem alterações
  2026_03_31_tribuna_liberal.csv — sem alterações
  2026_04_01_tribuna_liberal.csv — sem alterações

Correção concluída.


In [1]:
import pandas as pd
import os
import glob

caminho = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'

arquivos = sorted(glob.glob(os.path.join(caminho, '*.csv')))
print(f"Arquivos encontrados: {len(arquivos)}")

tipos_positivos = ['politica_publica', 'infraestrutura', 'institucional']

for arquivo in arquivos:
    df = pd.read_csv(arquivo, sep=None, engine='python', encoding='utf-8-sig')
    
    if 'polaridade_evento' in df.columns:
        antes = (df['polaridade_evento'] == 'positivo').sum()
        df.loc[df['tipo_evento'].isin(tipos_positivos), 'polaridade_evento'] = 'positivo'
        depois = (df['polaridade_evento'] == 'positivo').sum()
        corrigidos = depois - antes
        print(f"✓ {os.path.basename(arquivo)} — {corrigidos} registro(s) corrigido(s)")
    else:
        print(f"⚠️  {os.path.basename(arquivo)} — coluna polaridade_evento ausente")
    
    df.to_csv(arquivo, index=False, encoding='utf-8-sig', sep=',')

print("\nCorreção concluída.")

Arquivos encontrados: 11
✓ 2026_03_14_tribuna_liberal.csv — 0 registro(s) corrigido(s)
✓ 2026_03_18_tribuna_liberal.csv — 3 registro(s) corrigido(s)
✓ 2026_03_21_tribuna_liberal.csv — 4 registro(s) corrigido(s)
✓ 2026_03_24_tribuna_liberal.csv — 2 registro(s) corrigido(s)
✓ 2026_03_25_tribuna_liberal.csv — 3 registro(s) corrigido(s)
✓ 2026_03_26_tribuna_liberal.csv — 1 registro(s) corrigido(s)
✓ 2026_03_27_tribuna_liberal.csv — 3 registro(s) corrigido(s)
✓ 2026_03_28_tribuna_liberal.csv — 2 registro(s) corrigido(s)
✓ 2026_03_29_tribuna_liberal.csv — 3 registro(s) corrigido(s)
✓ 2026_03_31_tribuna_liberal.csv — 0 registro(s) corrigido(s)
✓ 2026_04_01_tribuna_liberal.csv — 0 registro(s) corrigido(s)

Correção concluída.


In [1]:
import pandas as pd
import os

caminho_csv = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'
nome_arquivo = '2026_04_01_tribuna_liberal.csv'
caminho_completo = os.path.join(caminho_csv, nome_arquivo)

colunas = [
    'fonte', 'data', 'item', 'pagina', 'dimensao_ivs', 'codigo_variavel',
    'tipo_relacao_variavel', 'resumo', 'localidade', 'tipo_evento',
    'gravidade', 'observacao', 'cod_loteamento', 'nivel_confianca_loteamento'
]

registros = [
    ['Tribuna Liberal', '2026-04-01',
     'Hortolandia conquista Selo FNAS 2025 na area de assistencia social',
     8, 'governanca', 'multidimensional', 'contextual_positivo',
     'Municipio conquista pela primeira vez o Selo FNAS 2025; rede SUAS atende 30 mil familias equivalente a 72 mil pessoas por meio de CRAS CREAS Centro Pop Casa de Passagem e Abrigo Institucional; reconhecimento federal por excelencia na gestao orcamentaria financeira contabil e regulatoria do Fundo Municipal de Assistencia Social',
     'Hortolandia', 'institucional', 'nao_aplicavel',
     'Evidencia de capacidade institucional instalada; suporte empirico para interpretacao IVS alto mais IPST baixo; gestao funcional pode estar absorvendo pressao territorial; primeiro selo conquistado pelo municipio; secretaria Maria dos Anjos citada',
     'nao_aplicavel', 'nao_aplicavel'],

    ['Tribuna Liberal', '2026-04-01',
     'GM apreende drogas em abrigo e mulher e detida em Hortolandia',
     8, 'capital_humano', 'CH_05', 'direta',
     'Mulher acolhida no Abrigo Esperancar com quatro filhos detida com 28 porcoes de cocaina e 15 de maconha; cao farejador Thor localizou entorpecentes sob movel no dormitorio feminino; Conselho Tutelar acionado para as criancas',
     'Hortolandia', 'vulnerabilidade', 'alta',
     'Criancas em situacao de risco dentro de equipamento de protecao social; interface CH_05 e vulnerabilidade infantil multipla; Jardim Conceicao RP_5; abrigo como loteamento de referencia',
     57, 'alto'],

    ['Tribuna Liberal', '2026-04-01',
     'Homem baleado e morto em madeireira no Jardim Amanda II',
     9, 'multidimensional', 'CH_01', 'indireta',
     'Eduardo Zani de 46 anos baleado na cabeca em madeireira no Jardim Amanda II; socorro realizado por populares em veiculo ate UPA Jardim Amanda; vitima nao resistiu antes de chegar a unidade',
     'Hortolandia', 'violencia', 'alta',
     'Homicidio em ambiente de trabalho informal; multidimensional CH mais RT indireto; ausencia de socorro institucional imediato; Jardim Amanda Quadra 1 RP_3',
     457, 'medio'],

    ['Tribuna Liberal', '2026-04-01',
     'Irmaos autuados apos sequencia de roubos de motos no Jardim Conceicao',
     9, 'capital_humano', 'CH_08', 'direta',
     'Dois irmaos detidos pela PM apos sequencia de roubos de motocicletas; adolescente entre os suspeitos apreendido; confessaram dois roubos incluindo Honda CB 500F e Kawasaki Versys',
     'Hortolandia', 'violencia', 'alta',
     'CH direto adolescente em conflito com a lei CH_08; RT contextual pressao socioeconomica indireta; Jardim Conceicao RP_5',
     57, 'medio'],
]

df_novo = pd.DataFrame(registros, columns=colunas)

# Verificar separador de um arquivo existente antes de salvar
df_ref = pd.read_csv(
    os.path.join(caminho_csv, '2026_03_31_tribuna_liberal.csv'),
    nrows=1, sep=None, engine='python'
)
sep_detectado = ','  # fallback

df_novo.to_csv(caminho_completo, index=False, encoding='utf-8-sig', sep=sep_detectado)

print(f"Arquivo salvo: {caminho_completo}")
print(f"Registros: {len(df_novo)}")
df_novo

Arquivo salvo: C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral\2026_04_01_tribuna_liberal.csv
Registros: 4


,fonte,data,item,pagina,dimensao_ivs,codigo_variavel,tipo_relacao_variavel,resumo,localidade,tipo_evento,gravidade,observacao,cod_loteamento,nivel_confianca_loteamento
0,Tribuna Liberal,2026-04-01,Hortolandia conquista Selo FNAS 2025 na area d...,8,governanca,multidimensional,contextual_positivo,Municipio conquista pela primeira vez o Selo F...,Hortolandia,institucional,nao_aplicavel,Evidencia de capacidade institucional instalad...,nao_aplicavel,nao_aplicavel
1,Tribuna Liberal,2026-04-01,GM apreende drogas em abrigo e mulher e detida...,8,capital_humano,CH_05,direta,Mulher acolhida no Abrigo Esperancar com quatr...,Hortolandia,vulnerabilidade,alta,Criancas em situacao de risco dentro de equipa...,57,alto
2,Tribuna Liberal,2026-04-01,Homem baleado e morto em madeireira no Jardim ...,9,multidimensional,CH_01,indireta,Eduardo Zani de 46 anos baleado na cabeca em m...,Hortolandia,violencia,alta,Homicidio em ambiente de trabalho informal; mu...,457,medio
3,Tribuna Liberal,2026-04-01,Irmaos autuados apos sequencia de roubos de mo...,9,capital_humano,CH_08,direta,Dois irmaos detidos pela PM apos sequencia de ...,Hortolandia,violencia,alta,CH direto adolescente em conflito com a lei CH...,57,medio


In [1]:
import pandas as pd
import os

# verificar onde o notebook está rodando
print("Diretório atual:", os.getcwd())

# caminho relativo correto
caminho = "../dados/tudo.csv"

# verificar se o arquivo existe
if os.path.exists(caminho):
    print("Arquivo encontrado!")
else:
    print("ERRO: arquivo NÃO encontrado!")

# leitura do CSV
df = pd.read_csv(
    caminho,
    sep=';',          # separador do CadÚnico
    encoding='utf-8',
    low_memory=False
)

print("Arquivo carregado com sucesso!")

Diretório atual: C:\Users\ailto\cadunico_projeto\notebooks
Arquivo encontrado!
Arquivo carregado com sucesso!


In [2]:
df.shape

(72424, 211)

In [3]:
df.columns.tolist()

['d.cd_ibge',
 ' d.cod_familiar_fam',
 ' d.dat_cadastramento_fam',
 ' d.dat_atual_fam',
 ' d.cod_est_cadastral_fam',
 ' d.cod_forma_coleta_fam',
 ' d.dta_entrevista_fam',
 ' d.nom_localidade_fam',
 ' d.nom_tip_logradouro_fam',
 ' d.nom_titulo_logradouro_fam',
 ' d.nom_logradouro_fam',
 ' d.num_logradouro_fam',
 ' d.des_complemento_fam',
 ' d.des_complemento_adic_fam',
 ' d.num_cep_logradouro_fam',
 ' d.cod_unidade_territorial_fam',
 ' d.nom_unidade_territorial_fam',
 ' d.txt_referencia_local_fam',
 ' d.nom_entrevistador_fam',
 ' d.num_cpf_entrevistador_fam',
 ' d.vlr_renda_media_fam',
 ' d.fx_rfpc',
 ' d.vlr_renda_total_fam',
 ' d.marc_pbf',
 ' d.qtde_meses_desat_cat',
 ' d.cod_local_domic_fam',
 ' d.cod_especie_domic_fam',
 ' d.qtd_comodos_domic_fam',
 ' d.qtd_comodos_dormitorio_fam',
 ' d.cod_material_piso_fam',
 ' d.cod_material_domic_fam',
 ' d.cod_agua_canalizada_fam',
 ' d.cod_abaste_agua_domic_fam',
 ' d.cod_banheiro_domic_fam',
 ' d.cod_escoa_sanitario_domic_fam',
 ' d.cod_dest

In [4]:
df.head()

,d.cd_ibge,d.cod_familiar_fam,d.dat_cadastramento_fam,d.dat_atual_fam,d.cod_est_cadastral_fam,d.cod_forma_coleta_fam,d.dta_entrevista_fam,d.nom_localidade_fam,d.nom_tip_logradouro_fam,d.nom_titulo_logradouro_fam,...,p.ind_dinh_carregador_memb,p.ind_dinh_catador_memb,p.ind_dinh_servs_gerais_memb,p.ind_dinh_pede_memb,p.ind_dinh_vendas_memb,p.ind_dinh_outro_memb,p.ind_dinh_nao_resp_memb,p.ind_atend_nenhum_memb,p.ref_cad,p.ref_pbf
0,3519071,8477396,17/12/2001,04/08/2025,3,1,04/08/2025,JARDIM MINDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0
1,3519071,8477396,17/12/2001,04/08/2025,3,1,04/08/2025,JARDIM MINDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0
2,3519071,8477396,17/12/2001,04/08/2025,3,1,04/08/2025,JARDIM MINDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0
3,3519071,8477396,17/12/2001,04/08/2025,3,1,04/08/2025,JARDIM MINDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0
4,3519071,37372840,07/02/2002,14/04/2025,3,1,14/04/2025,JARDIM MONTE SINAI,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0


In [5]:
unidades_assistencia = df[' d.nom_unidade_territorial_fam'].dropna().unique()

len(unidades_assistencia)

12

In [6]:
df[' d.cod_familiar_fam'].nunique()

30362

In [7]:
df_unidades = pd.DataFrame(unidades_assistencia, columns=[' unidade_assistencia_bruta'])

df_unidades.to_csv("../outputs/unidades_assistencia_brutas.csv", index=False)

🔍 Achados Principais & Preliminares na análise do CadÚnico de Dezembro de 2025

A análise exploratória revelou que o campo não representa núcleos geográficos puro, mas sim uma estrutura híbrida, contendo:

🟢 Unidades de Proteção Social Básica
CRAS AMANDA
CRAS SANTA CLARA
CRAS NOVO ÂNGULO
etc.
🟡 Unidades de Proteção Social Especial
CREAS
🔴 Equipamentos específicos
CENTRO POP
🟠 Variações nominais e granularidade inconsistente
CRAS PRIMAVERA
CRAS PRIMAVERA CHICO VIGILANTE
CRAS ROSOLEN
CRAS ROSOLEN JOEL ALVES ASSUNCAO
⚠️ Diagnóstico Técnico

O campo d.nom_unidade_territorial_fam apresenta:

Mistura de unidade administrativa e território
Ausência de padronização textual
Inclusão de nomes institucionais e patronímicos
Variação de granularidade (macro vs microterritório)

👉 Portanto, trata-se de um:

Campo semântico híbrido (não normalizado)

📊 Evidência Quantitativa
Total de unidades distintas identificadas: 12
Número esperado de CRAS no município: 7

👉 Indica presença de:

duplicidade semântica
sobreposição de categorias
ruído estrutural no dado
🧠 Interpretação Conceitual

O campo representa, na prática:

Núcleo operacional da assistência social, e não núcleo geográfico formal.

Ou seja:

Reflete a lógica de atendimento do SUAS
Não corresponde diretamente a:
bairros
loteamentos
divisões do IBGE

In [8]:
df.columns = df.columns.str.strip()
print(df.columns.tolist())

['d.cd_ibge', 'd.cod_familiar_fam', 'd.dat_cadastramento_fam', 'd.dat_atual_fam', 'd.cod_est_cadastral_fam', 'd.cod_forma_coleta_fam', 'd.dta_entrevista_fam', 'd.nom_localidade_fam', 'd.nom_tip_logradouro_fam', 'd.nom_titulo_logradouro_fam', 'd.nom_logradouro_fam', 'd.num_logradouro_fam', 'd.des_complemento_fam', 'd.des_complemento_adic_fam', 'd.num_cep_logradouro_fam', 'd.cod_unidade_territorial_fam', 'd.nom_unidade_territorial_fam', 'd.txt_referencia_local_fam', 'd.nom_entrevistador_fam', 'd.num_cpf_entrevistador_fam', 'd.vlr_renda_media_fam', 'd.fx_rfpc', 'd.vlr_renda_total_fam', 'd.marc_pbf', 'd.qtde_meses_desat_cat', 'd.cod_local_domic_fam', 'd.cod_especie_domic_fam', 'd.qtd_comodos_domic_fam', 'd.qtd_comodos_dormitorio_fam', 'd.cod_material_piso_fam', 'd.cod_material_domic_fam', 'd.cod_agua_canalizada_fam', 'd.cod_abaste_agua_domic_fam', 'd.cod_banheiro_domic_fam', 'd.cod_escoa_sanitario_domic_fam', 'd.cod_destino_lixo_domic_fam', 'd.cod_iluminacao_domic_fam', 'd.cod_calcamento_d

In [9]:
print('unidade_raw' in df.columns)
print('unidade_norm' in df.columns)
print('unidade_oficial' in df.columns)

False
False
False


In [10]:
df.columns = df.columns.str.strip()
for c in df.columns:
    print(repr(c))

'd.cd_ibge'
'd.cod_familiar_fam'
'd.dat_cadastramento_fam'
'd.dat_atual_fam'
'd.cod_est_cadastral_fam'
'd.cod_forma_coleta_fam'
'd.dta_entrevista_fam'
'd.nom_localidade_fam'
'd.nom_tip_logradouro_fam'
'd.nom_titulo_logradouro_fam'
'd.nom_logradouro_fam'
'd.num_logradouro_fam'
'd.des_complemento_fam'
'd.des_complemento_adic_fam'
'd.num_cep_logradouro_fam'
'd.cod_unidade_territorial_fam'
'd.nom_unidade_territorial_fam'
'd.txt_referencia_local_fam'
'd.nom_entrevistador_fam'
'd.num_cpf_entrevistador_fam'
'd.vlr_renda_media_fam'
'd.fx_rfpc'
'd.vlr_renda_total_fam'
'd.marc_pbf'
'd.qtde_meses_desat_cat'
'd.cod_local_domic_fam'
'd.cod_especie_domic_fam'
'd.qtd_comodos_domic_fam'
'd.qtd_comodos_dormitorio_fam'
'd.cod_material_piso_fam'
'd.cod_material_domic_fam'
'd.cod_agua_canalizada_fam'
'd.cod_abaste_agua_domic_fam'
'd.cod_banheiro_domic_fam'
'd.cod_escoa_sanitario_domic_fam'
'd.cod_destino_lixo_domic_fam'
'd.cod_iluminacao_domic_fam'
'd.cod_calcamento_domic_fam'
'd.cod_familia_indigena_fam'

In [11]:
[col for col in df.columns if 'unidad' in c.lower() or 'equip' in c.lower() or 'cras' in c.lower()]

[]

In [12]:
df['p.ind_atend_cras_memb'].value_counts(dropna=False)

p.ind_atend_cras_memb
NaN    71995
0.0      353
1.0       76
Name: count, dtype: int64

In [13]:
for col in df.columns:
    if df[col].nunique() <= 20:
        print(col, df[col].nunique())

d.cd_ibge 1
d.cod_est_cadastral_fam 1
d.cod_forma_coleta_fam 3
d.cod_unidade_territorial_fam 8
d.nom_unidade_territorial_fam 12
d.fx_rfpc 4
d.marc_pbf 2
d.qtde_meses_desat_cat 5
d.cod_local_domic_fam 2
d.cod_especie_domic_fam 3
d.qtd_comodos_domic_fam 20
d.qtd_comodos_dormitorio_fam 12
d.cod_material_piso_fam 7
d.cod_material_domic_fam 7
d.cod_agua_canalizada_fam 2
d.cod_abaste_agua_domic_fam 4
d.cod_banheiro_domic_fam 2
d.cod_escoa_sanitario_domic_fam 6
d.cod_destino_lixo_domic_fam 6
d.cod_iluminacao_domic_fam 5
d.cod_calcamento_domic_fam 3
d.cod_familia_indigena_fam 2
d.cod_povo_indigena_fam 3
d.nom_povo_indigena_fam 3
d.cod_indigena_reside_fam 1
d.cod_reserva_indigena_fam 0
d.nom_reserva_indigena_fam 0
d.ind_familia_quilombola_fam 1
d.cod_comunidade_quilombola_fam 0
d.nom_comunidade_quilombola_fam 0
d.qtd_pessoas_domic_fam 14
d.qtd_familias_domic_fam 8
d.qtd_pessoa_inter_0_17_anos_fam 4
d.qtd_pessoa_inter_18_64_anos_fam 4
d.qtd_pessoa_inter_65_anos_fam 3
d.nom_centro_assist_fam 12
d

In [14]:
df['d.nom_unidade_territorial_fam'].value_counts()

d.nom_unidade_territorial_fam
CRAS AMANDA                                      16618
CRAS SANTA CLARA                                 14700
CRAS NOVO ANGULO                                 13752
CRAS PRIMAVERA                                   12687
CRAS PRIMAVERA CHICO VIGILANTE                    5761
CRAS ZUMA JD BRASIL                               3219
CRAS JARDIM BRASIL MARIA HUMILDE ANTUNES ZUMA     1927
CRAS ROSOLEN JOEL ALVES ASSUNCAO                  1764
CRAS ROSOLEN                                       784
CRAS VILA REAL                                     522
CENTRO POP                                         241
CREAS                                                6
Name: count, dtype: int64

In [15]:
[col for col in df.columns if 'cpf' in col.lower()]

['d.num_cpf_entrevistador_fam', 'p.num_cpf_pessoa']

In [16]:
df['p.num_cpf_pessoa'].nunique()

71838

In [17]:
df['p.num_cpf_pessoa'].isna().sum()

586

In [18]:
df['p.num_cpf_pessoa'].notna().sum()

71838

In [19]:
df[df.duplicated('p.num_cpf_pessoa', keep=False)] \
    .sort_values('p.num_cpf_pessoa') \
    .head(20)

,d.cd_ibge,d.cod_familiar_fam,d.dat_cadastramento_fam,d.dat_atual_fam,d.cod_est_cadastral_fam,d.cod_forma_coleta_fam,d.dta_entrevista_fam,d.nom_localidade_fam,d.nom_tip_logradouro_fam,d.nom_titulo_logradouro_fam,...,p.ind_dinh_carregador_memb,p.ind_dinh_catador_memb,p.ind_dinh_servs_gerais_memb,p.ind_dinh_pede_memb,p.ind_dinh_vendas_memb,p.ind_dinh_outro_memb,p.ind_dinh_nao_resp_memb,p.ind_atend_nenhum_memb,p.ref_cad,p.ref_pbf
11,3519071,47959118,01/03/2002,06/09/2024,3,1,06/09/2024,JARDIM RESIDENCIAL FIRENZE,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0
187,3519071,485610108,24/10/2002,11/09/2025,3,1,11/09/2025,LOTEAMENTO RECANTO DO SOL,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0
266,3519071,529442124,28/11/2002,13/09/2024,3,1,13/09/2024,JARDIM AMANDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0
447,3519071,649625129,31/03/2003,13/05/2022,3,1,13/05/2022,JARDIM AMANDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,NaN
732,3519071,1198411341,12/09/2003,19/07/2024,3,1,19/07/2024,JARDIM SAO FELIPE,AVENIDA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0
765,3519071,1198416815,12/09/2003,20/12/2022,3,1,20/12/2022,JARDIM NOVO ESTRELA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,NaN
774,3519071,1208566547,16/10/2003,21/01/2025,3,2,20/01/2025,JARDIM SUMAREZINHO,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0
842,3519071,1275423108,28/01/2004,11/11/2025,3,1,11/11/2025,JARDIM NOVO HORIZONTE,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,NaN
999,3519071,1378871596,01/07/2004,05/09/2025,3,2,05/09/2025,AMANDA 1,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,NaN
1089,3519071,1394013507,16/07/2004,14/05/2024,3,1,14/05/2024,JARDIM AMANDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0


In [20]:
df['p.num_cpf_pessoa'] = df['p.num_cpf_pessoa'].astype(str).str.strip()

In [21]:
df['p.num_cpf_pessoa'].value_counts().head(10)

p.num_cpf_pessoa
nan              586
92194290397.0      1
37196118847.0      1
61046987860.0      1
50476214807.0      1
21459740831.0      1
96572710882.0      1
50570177804.0      1
12065372800.0      1
49053736549.0      1
Name: count, dtype: int64

In [22]:
df['cpf_tratado'] = (
    df['p.num_cpf_pessoa']
    .str.replace('.0', '', regex=False)
)

In [23]:
df['cpf_tratado'] = df['cpf_tratado'].replace('nan', None)

In [24]:
df['cpf_tratado'] = df['cpf_tratado'].str.zfill(11)

In [25]:
df['cpf_tratado'].value_counts().head(10)

cpf_tratado
92194290397    1
49053736549    1
61046987860    1
50476214807    1
21459740831    1
96572710882    1
50570177804    1
12065372800    1
53523118808    1
37196118847    1
Name: count, dtype: int64

In [26]:
df.loc[df['cpf_tratado'] == '00000000000', 'cpf_tratado'] = None

In [27]:
df.shape

(72424, 212)

In [28]:
df.head(9)

,d.cd_ibge,d.cod_familiar_fam,d.dat_cadastramento_fam,d.dat_atual_fam,d.cod_est_cadastral_fam,d.cod_forma_coleta_fam,d.dta_entrevista_fam,d.nom_localidade_fam,d.nom_tip_logradouro_fam,d.nom_titulo_logradouro_fam,...,p.ind_dinh_catador_memb,p.ind_dinh_servs_gerais_memb,p.ind_dinh_pede_memb,p.ind_dinh_vendas_memb,p.ind_dinh_outro_memb,p.ind_dinh_nao_resp_memb,p.ind_atend_nenhum_memb,p.ref_cad,p.ref_pbf,cpf_tratado
0,3519071,8477396,17/12/2001,04/08/2025,3,1,04/08/2025,JARDIM MINDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0,92194290397
1,3519071,8477396,17/12/2001,04/08/2025,3,1,04/08/2025,JARDIM MINDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0,08634143384
2,3519071,8477396,17/12/2001,04/08/2025,3,1,04/08/2025,JARDIM MINDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0,07660349333
3,3519071,8477396,17/12/2001,04/08/2025,3,1,04/08/2025,JARDIM MINDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0,08634164381
4,3519071,37372840,07/02/2002,14/04/2025,3,1,14/04/2025,JARDIM MONTE SINAI,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0,01144328306
5,3519071,37372840,07/02/2002,14/04/2025,3,1,14/04/2025,JARDIM MONTE SINAI,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0,59779029800
6,3519071,46512861,27/02/2002,08/11/2025,3,1,16/06/2025,JARDIM NOVA EUROPA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0,04919933401
7,3519071,46512861,27/02/2002,08/11/2025,3,1,16/06/2025,JARDIM NOVA EUROPA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0,29508006889
8,3519071,46512861,27/02/2002,08/11/2025,3,1,16/06/2025,JARDIM NOVA EUROPA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0,53687444805


In [29]:
df.columns.tolist()

['d.cd_ibge',
 'd.cod_familiar_fam',
 'd.dat_cadastramento_fam',
 'd.dat_atual_fam',
 'd.cod_est_cadastral_fam',
 'd.cod_forma_coleta_fam',
 'd.dta_entrevista_fam',
 'd.nom_localidade_fam',
 'd.nom_tip_logradouro_fam',
 'd.nom_titulo_logradouro_fam',
 'd.nom_logradouro_fam',
 'd.num_logradouro_fam',
 'd.des_complemento_fam',
 'd.des_complemento_adic_fam',
 'd.num_cep_logradouro_fam',
 'd.cod_unidade_territorial_fam',
 'd.nom_unidade_territorial_fam',
 'd.txt_referencia_local_fam',
 'd.nom_entrevistador_fam',
 'd.num_cpf_entrevistador_fam',
 'd.vlr_renda_media_fam',
 'd.fx_rfpc',
 'd.vlr_renda_total_fam',
 'd.marc_pbf',
 'd.qtde_meses_desat_cat',
 'd.cod_local_domic_fam',
 'd.cod_especie_domic_fam',
 'd.qtd_comodos_domic_fam',
 'd.qtd_comodos_dormitorio_fam',
 'd.cod_material_piso_fam',
 'd.cod_material_domic_fam',
 'd.cod_agua_canalizada_fam',
 'd.cod_abaste_agua_domic_fam',
 'd.cod_banheiro_domic_fam',
 'd.cod_escoa_sanitario_domic_fam',
 'd.cod_destino_lixo_domic_fam',
 'd.cod_ilumin

In [30]:
df['d.cod_familiar_fam'].nunique()

30362

In [31]:
df.dtypes

d.cd_ibge                     int64
d.cod_familiar_fam            int64
d.dat_cadastramento_fam      object
d.dat_atual_fam              object
d.cod_est_cadastral_fam       int64
                             ...   
p.ind_dinh_nao_resp_memb    float64
p.ind_atend_nenhum_memb     float64
p.ref_cad                    object
p.ref_pbf                   float64
cpf_tratado                  object
Length: 212, dtype: object

In [32]:
df['data_cadastro'] = pd.to_datetime(
    df['d.dat_cadastramento_fam'],
    dayfirst=True,
    errors='coerce'
)

In [33]:
df['ano_cadastro'] = df['data_cadastro'].dt.year

In [34]:
familias_por_ano = (
    df.groupby('ano_cadastro')['d.cod_familiar_fam']
      .nunique()
      .reset_index()
      .sort_values('ano_cadastro')
)

In [35]:
print(familias_por_ano)

    ano_cadastro  d.cod_familiar_fam
0           2000                   1
1           2001                   4
2           2002                 165
3           2003                 177
4           2004                 207
5           2005                 256
6           2006                 341
7           2007                 267
8           2008                 252
9           2009                 216
10          2010                 401
11          2011                 445
12          2012                 642
13          2013                 533
14          2014                 697
15          2015                 727
16          2016                1041
17          2017                1616
18          2018                1983
19          2019                1698
20          2020                1312
21          2021                1994
22          2022                5240
23          2023                3508
24          2024                4267
25          2025                2372


In [36]:
df['d.nom_localidade_fam'].value_counts().head(30)

d.nom_localidade_fam
JARDIM AMANDA                       10208
JARDIM BOA ESPERANCA                 2564
JARDIM NOVO ANGULO                   2350
JARDIM NOVA EUROPA                   2018
JARDIM NOVA HORTOLANDIA              1937
JARDIM NOVA AMERICA                  1854
JARDIM NOSSA SENHORA AUXILIADORA     1427
JARDIM ADELAIDE                      1364
JARDIM SAO SEBASTIAO                 1227
VILA REAL CONTINUACAO                1225
JARDIM AMANDA 2                      1204
REMANSO CAMPINEIRO                   1189
JARDIM SANTA CLARA DO LAGO           1161
PARQUE DO HORTO                      1138
JARDIM AMANDA II                     1133
JARDIM NOSSA SENHORA DE FATIMA       1131
JARDIM PRIMAVERA                     1121
JARDIM CAMPOS VERDES                 1118
JARDIM SUMAREZINHO                   1074
NOVO ANGULO                          1065
JARDIM MINDA                         1030
JARDIM SAO BENTO                     1017
JARDIM SAO JORGE                      982
VILA REAL    

In [37]:
familias_por_territorio = (
    df.groupby('d.nom_localidade_fam')['d.cod_familiar_fam']
      .nunique()
      .reset_index()
      .sort_values('d.cod_familiar_fam', ascending=False)
)

In [51]:
print(familias_por_territorio.head(60))

                 d.nom_localidade_fam  d.cod_familiar_fam
104                     JARDIM AMANDA                4325
129              JARDIM BOA ESPERANCA                1028
278                JARDIM NOVO ANGULO                 961
260                JARDIM NOVA EUROPA                 820
266           JARDIM NOVA HORTOLANDIA                 807
254               JARDIM NOVA AMERICA                 770
575                REMANSO CAMPINEIRO                 692
237  JARDIM NOSSA SENHORA AUXILIADORA                 592
93                    JARDIM ADELAIDE                 575
656             VILA REAL CONTINUACAO                 543
320        JARDIM SANTA CLARA DO LAGO                 526
245    JARDIM NOSSA SENHORA DE FATIMA                 511
367              JARDIM SAO SEBASTIAO                 500
108                   JARDIM AMANDA 2                 495
138              JARDIM CAMPOS VERDES                 493
382                JARDIM SUMAREZINHO                 478
110           

In [39]:
familias_por_territorio.head(30)

,d.nom_localidade_fam,d.cod_familiar_fam
104,JARDIM AMANDA,4325
129,JARDIM BOA ESPERANCA,1028
278,JARDIM NOVO ANGULO,961
260,JARDIM NOVA EUROPA,820
266,JARDIM NOVA HORTOLANDIA,807
254,JARDIM NOVA AMERICA,770
575,REMANSO CAMPINEIRO,692
237,JARDIM NOSSA SENHORA AUXILIADORA,592
93,JARDIM ADELAIDE,575
656,VILA REAL CONTINUACAO,543


In [40]:
df['territorio_simples'] = (
    df['d.nom_localidade_fam']
    .str.replace(' I$', '', regex=True)
    .str.replace(' II$', '', regex=True)
    .str.replace(' 1$', '', regex=True)
    .str.replace(' 2$', '', regex=True)
)

In [41]:
familias_territorio_ano.sort_values(
    ['ano_cadastro', 'd.cod_familiar_fam'],
    ascending=[True, False]
).head(20)

NameError: name 'familias_territorio_ano' is not defined

In [42]:
'familias_territorio_ano' in globals()

False

In [43]:
familias_territorio_ano = (
    df.groupby(['ano_cadastro', 'territorio_simples'])['d.cod_familiar_fam']
      .nunique()
      .reset_index()
)

print(familias_territorio_ano.head(10))

   ano_cadastro      territorio_simples  d.cod_familiar_fam
0          2000           JARDIM AMANDA                   1
1          2001  JARDIM CARMEN CRISTINA                   1
2          2001        JARDIM ESTEFANIA                   1
3          2001            JARDIM MINDA                   1
4          2001        JARDIM SAO BENTO                   1
5          2002  CHACARA FAZENDA COELHO                   1
6          2002          CHACARA REYMAR                   1
7          2002          CHACARAS ASSAY                   1
8          2002         JARDIM ADELAIDE                   3
9          2002           JARDIM AMANDA                  54


In [44]:
[var for var in globals() if 'territorio' in var or 'familias' in var]

['familias_por_ano', 'familias_por_territorio', 'familias_territorio_ano']

In [45]:
'familias_territorio_ano' in globals()

True

In [46]:
[var for var in globals() if 'territorio' in var or 'familias' in var]

['familias_por_ano', 'familias_por_territorio', 'familias_territorio_ano']

In [5]:
import pandas as pd
import glob
import os

In [6]:
pasta = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'

arquivos = glob.glob(pasta + r'\2026_*.csv')

print(f'Arquivos Encontrados: {len(arquivos)}')
for a in arquivos:
    print(os.path.basename(a))

Arquivos Encontrados: 10
2026_03_14_tribuna_liberal.csv
2026_03_18_tribuna_liberal.csv
2026_03_21_tribuna_liberal.csv
2026_03_24_tribuna_liberal.csv
2026_03_25_tribuna_liberal.csv
2026_03_26_tribuna_liberal.csv
2026_03_27_tribuna_liberal.csv
2026_03_28_tribuna_liberal.csv
2026_03_29_tribuna_liberal.csv
2026_03_31_tribuna_liberal.csv


In [5]:
df = pd.concat(
    [pd.read_csv(f, sep=',', encoding='utf-8') for f in arquivos],
    ignore_index=True
)

print(f'Total de registros: {len(df)}')
print(f'Colunas: {list(df.columns)}')

Total de registros: 38
Colunas: ['fonte', 'data', 'item', 'pagina', 'dimensao_ivs', 'codigo_variavel', 'tipo_relacao_variavel', 'resumo', 'localidade', 'tipo_evento', 'gravidade', 'observacao\t\t\t', 'cod_loteamento', 'nivel_confianca_loteamento', 'polaridade_evento', 'tipo_origem_evento', 'observacao\t', 'observacao', 'observacao\t\t']


Linha a linha:

pd.read_csv(f, sep=',', encoding='utf-8') → lê um arquivo CSV como tabela
[... for f in arquivos] → list comprehension: para cada arquivo da lista, lê e cria uma tabela — isso substitui um for com 3 linhas por 1 linha
pd.concat(...) → empilha todas as tabelas em uma só
ignore_index=True → reinicia a numeração das linhas de 1 até o total
len(df) → número de linhas
df.columns → nomes das colunas

In [8]:
df.head()

,fonte,data,item,pagina,dimensao_ivs,codigo_variavel,tipo_relacao_variavel,resumo,localidade,tipo_evento,gravidade,observacao\t\t\t,cod_loteamento,nivel_confianca_loteamento,observacao\t,observacao,observacao\t\t
0,Tribuna Liberal,2026-03-14,Medidas protetivas,s/n,capital_humano,CH_04,direta,94 medidas protetivas em jan-fev/2026\t maior ...,Hortolandia,problema,alta,Fonte TJ-SP a pedido do jornal\t dado auditave...,NaN,NaN,NaN,NaN,NaN
1,Tribuna Liberal,2026-03-15,Caso Nicolly Pogere,s/n,capital_humano,CH_03,direta,Adolescente assassinada em jul/2025\t casal de...,Hortolandia,caso_individual,alta,Interface CREAS e Conselho Tutelar\t impacta C...,NaN,NaN,NaN,NaN,NaN
2,Tribuna Liberal,2026-03-18,Forum de Empregabilidade,s/n,renda_trabalho,RT_02,direta,Forum Dimas + Maria dos Anjos\t vinculado ao p...,Hortolandia,politica_publica,media,NaN,NaN,NaN,Dado registrado em dim_programa_v05,NaN,NaN
3,Tribuna Liberal,2026-03-18,Embalixo 5x2,s/n,renda_trabalho,RT_02,direta,110 contratacoes formais\t dado futuro para cr...,Hortolandia,politica_publica,media,NaN,NaN,NaN,Evidencia empirica de insercao produtiva formal,NaN,NaN
4,Tribuna Liberal,2026-03-18,Ministra das Mulheres Jardim Amanda,s/n,capital_humano,CH_04,direta,Visita ao nucleo Amanda\t contexto de vulnerab...,Jd Amanda,politica_publica,media,NaN,NaN,NaN,Nucleo mais presente no CadUnico com 10208 reg...,NaN,NaN


In [6]:
df['dimensao_ivs'].value_counts()

dimensao_ivs
capital_humano           16
renda_trabalho            7
infraestrutura_urbana     6
multidimensional          6
SMIDS_EXT                 2
governanca                1
Name: count, dtype: int64

In [7]:
df['tipo_relacao_variavel'].value_counts()

tipo_relacao_variavel
direta        26
contextual     8
indireta       4
Name: count, dtype: int64

In [12]:
df_diretos = df[df['tipo_relacao_variavel'] == 'direta']

print(f'Registros Diretos: {len(df_diretos)}')
df_diretos[['data', 'item', 'codigo_variavel', 'localidade']].head(10)

Registros Diretos: 24


,data,item,codigo_variavel,localidade
0,2026-03-14,Medidas protetivas,CH_04,Hortolandia
1,2026-03-15,Caso Nicolly Pogere,CH_03,Hortolandia
2,2026-03-18,Forum de Empregabilidade,RT_02,Hortolandia
3,2026-03-18,Embalixo 5x2,RT_02,Hortolandia
4,2026-03-18,Ministra das Mulheres Jardim Amanda,CH_04,Jd Amanda
5,2026-03-21,Exposicao Mulheres que Inspiram,CH_04,Hortolandia
7,2026-03-21,17a Caravana Federativa equipamentos saude,CH_01,Hortolandia
8,2026-03-21,"Reajuste funcionalismo 2,51%",RT_01,Hortolandia
9,2026-03-24,Mercado imobiliario +10%,IU_01,Hortolandia
10,2026-03-24,Parque Terras de Santa Maria,IU_01,Hortolandia


df[df['coluna'] == 'valor'] → filtra linhas onde a condição é verdadeira — equivale ao filtro do Excel
df_diretos[['data', 'item', ...]] → seleciona só algumas colunas para exibir

In [8]:
df.columns.tolist()

['fonte',
 'data',
 'item',
 'pagina',
 'dimensao_ivs',
 'codigo_variavel',
 'tipo_relacao_variavel',
 'resumo',
 'localidade',
 'tipo_evento',
 'gravidade',
 'observacao\t\t\t',
 'cod_loteamento',
 'nivel_confianca_loteamento',
 'polaridade_evento',
 'tipo_origem_evento',
 'observacao\t',
 'observacao',
 'observacao\t\t']

In [9]:
df['codigo_variavel'].value_counts()

codigo_variavel
CH_05               5
CH_04               4
CH_01               4
RT_01               4
IU_01               4
SMIDS_EXT           4
RT_04               3
RT_02               2
CH_08               2
CH_07               2
CH_03               1
IU_02               1
governanca          1
multidimensional    1
Name: count, dtype: int64

In [11]:
import pandas as pd
import os
import glob

caminho = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'
arquivos = sorted(glob.glob(os.path.join(caminho, '*.csv')))

for arquivo in arquivos:
    df = pd.read_csv(arquivo, sep=',', engine='python', encoding='utf-8-sig')
    
    # Identificar todas as colunas com 'observacao' no nome
    colunas_obs = [c for c in df.columns if 'observacao' in c.lower()]
    
    if len(colunas_obs) > 1:
        # Consolidar todas em uma única coluna 'observacao'
        df['observacao'] = df[colunas_obs].apply(
            lambda row: ' '.join([str(v) for v in row if pd.notna(v) and str(v).strip() != '']),
            axis=1
        )
        # Remover colunas fantasmas
        df = df.drop(columns=[c for c in colunas_obs if c != 'observacao'])
        print(f"✓ {os.path.basename(arquivo)} — colunas fantasmas removidas: {colunas_obs}")
    else:
        print(f"  {os.path.basename(arquivo)} — sem problema")
    
    df.to_csv(arquivo, index=False, encoding='utf-8-sig', sep=',')

print("\nLimpeza concluída.")

  2026_03_14_tribuna_liberal.csv — sem problema
  2026_03_18_tribuna_liberal.csv — sem problema
  2026_03_21_tribuna_liberal.csv — sem problema
  2026_03_24_tribuna_liberal.csv — sem problema
  2026_03_25_tribuna_liberal.csv — sem problema
  2026_03_26_tribuna_liberal.csv — sem problema
  2026_03_27_tribuna_liberal.csv — sem problema
  2026_03_28_tribuna_liberal.csv — sem problema
  2026_03_29_tribuna_liberal.csv — sem problema
  2026_03_31_tribuna_liberal.csv — sem problema
  2026_04_01_tribuna_liberal.csv — sem problema

Limpeza concluída.


In [14]:
df = pd.concat(
    [pd.read_csv(f, sep=',', encoding='utf-8') for f in arquivos],
    ignore_index=True
)

print(f'Total de registros: {len(df)}')
print(f'Colunas: {list(df.columns)}')

Total de registros: 38
Colunas: ['fonte', 'data', 'item', 'pagina', 'dimensao_ivs', 'codigo_variavel', 'tipo_relacao_variavel', 'resumo', 'localidade', 'tipo_evento', 'gravidade', 'observacao\t\t\t', 'cod_loteamento', 'nivel_confianca_loteamento', 'polaridade_evento', 'tipo_origem_evento', 'observacao\t', 'observacao', 'observacao\t\t']


In [15]:
df.columns.tolist()

['fonte',
 'data',
 'item',
 'pagina',
 'dimensao_ivs',
 'codigo_variavel',
 'tipo_relacao_variavel',
 'resumo',
 'localidade',
 'tipo_evento',
 'gravidade',
 'observacao\t\t\t',
 'cod_loteamento',
 'nivel_confianca_loteamento',
 'polaridade_evento',
 'tipo_origem_evento',
 'observacao\t',
 'observacao',
 'observacao\t\t']

In [23]:
df['codigo_variavel'].value_counts()

codigo_variavel
CH_05               5
CH_04               4
CH_01               4
RT_01               4
IU_01               4
SMIDS_EXT           4
RT_04               3
RT_02               2
CH_08               2
CH_07               2
CH_03               1
IU_02               1
governanca          1
multidimensional    1
Name: count, dtype: int64

In [24]:
# Renomear colunas com variações de 'observacao'
novas_colunas = []

for c in df.columns:
    if 'observacao' in c:
        novas_colunas.append('observacao')
    else:
        novas_colunas.append(c)

df.columns = novas_colunas

In [25]:
df.columns.tolist()

['fonte',
 'data',
 'item',
 'pagina',
 'dimensao_ivs',
 'codigo_variavel',
 'tipo_relacao_variavel',
 'resumo',
 'localidade',
 'tipo_evento',
 'gravidade',
 'observacao',
 'cod_loteamento',
 'nivel_confianca_loteamento',
 'polaridade_evento',
 'tipo_origem_evento']

In [26]:
df.columns.tolist()

['fonte',
 'data',
 'item',
 'pagina',
 'dimensao_ivs',
 'codigo_variavel',
 'tipo_relacao_variavel',
 'resumo',
 'localidade',
 'tipo_evento',
 'gravidade',
 'observacao',
 'cod_loteamento',
 'nivel_confianca_loteamento',
 'polaridade_evento',
 'tipo_origem_evento']

In [20]:
# Ver o conteúdo de cada coluna observacao
posicoes = [i for i, c in enumerate(df.columns) if c == 'observacao']
print(f"Posições encontradas: {posicoes}")

for pos in posicoes:
    print(f"\n--- Posição {pos} ---")
    valores = df.iloc[:, pos].dropna()
    print(f"Registros preenchidos: {len(valores)}")
    print(valores.unique())

Posições encontradas: [11, 16, 17, 18]

--- Posição 11 ---
Registros preenchidos: 2
['Fonte TJ-SP a pedido do jornal\t dado auditavel para CRAM_01 e CREAS_01'
 'Interface CREAS e Conselho Tutelar\t impacta CH_03 e CH_08']

--- Posição 16 ---
Registros preenchidos: 6
['Dado registrado em dim_programa_v05'
 'Evidencia empirica de insercao produtiva formal'
 'Nucleo mais presente no CadUnico com 10208 registros'
 'Acao institucional SMIDS' 'Cruzar com loteamentos afetados\t'
 'Caso concreto da variavel RT_04']

--- Posição 17 ---
Registros preenchidos: 23
['Acao pontual nao entra no catalogo de programas'
 'Impacto futuro em RT_04 Economia Prateada'
 'Amplia capacidade de resposta Capital Humano'
 'Contexto de custo de vida e pressao sobre renda'
 'Contexto político relevante para período de apresentação em abril'
 'Ação da Educação; interface com CH_03 e CH_07; presença do prefeito Zezé'
 'Parque Gabriel em área de vulnerabilidade; relevância territorial'
 'Ocorrência de segurança no Jd.

In [27]:
# Consolidar as 4 colunas observacao em uma só
posicoes = [i for i, c in enumerate(df.columns) if c == 'observacao']

# Pegar o primeiro valor não vazio entre as 4 colunas, linha a linha
df['observacao'] = df.iloc[:, posicoes].apply(
    lambda row: ' | '.join([str(v).strip() for v in row if pd.notna(v) and str(v).strip() != '']),
    axis=1
)

# Remover duplicatas — manter só a primeira ocorrência
df = df.loc[:, ~df.columns.duplicated()]

print(f"Colunas após limpeza: {len(df.columns)}")
print(df['observacao'].dropna().head(5))

Colunas após limpeza: 16
0    Fonte TJ-SP a pedido do jornal  dado auditavel...
1    Interface CREAS e Conselho Tutelar  impacta CH...
2                  Dado registrado em dim_programa_v05
3      Evidencia empirica de insercao produtiva formal
4    Nucleo mais presente no CadUnico com 10208 reg...
Name: observacao, dtype: object


In [28]:
# Remover tabs residuais dentro do texto
df['observacao'] = df['observacao'].str.replace('\t', ' ', regex=False)

# Confirmar
print(df['observacao'].head(5))

0    Fonte TJ-SP a pedido do jornal  dado auditavel...
1    Interface CREAS e Conselho Tutelar  impacta CH...
2                  Dado registrado em dim_programa_v05
3      Evidencia empirica de insercao produtiva formal
4    Nucleo mais presente no CadUnico com 10208 reg...
Name: observacao, dtype: object


In [29]:
print(df['observacao'].iloc[0])
print('---')
print(df['observacao'].iloc[1])

Fonte TJ-SP a pedido do jornal  dado auditavel para CRAM_01 e CREAS_01
---
Interface CREAS e Conselho Tutelar  impacta CH_03 e CH_08


In [30]:
import re
df['observacao'] = df['observacao'].str.replace(r' +', ' ', regex=True)

print(df['observacao'].iloc[0])
print('---')
print(df['observacao'].iloc[1])

Fonte TJ-SP a pedido do jornal dado auditavel para CRAM_01 e CREAS_01
---
Interface CREAS e Conselho Tutelar impacta CH_03 e CH_08


In [31]:
r'C:\Users\ailto\atlas_social_projeto\...'

'C:\\Users\\ailto\\atlas_social_projeto\\...'

In [32]:
import pandas as pd
import os
import glob

caminho = r'C:\Users\ailto\atlas_social_projeto\dados\bd_externos\imprensa_em_geral'
arquivos = sorted(glob.glob(os.path.join(caminho, '*.csv')))

for arquivo in arquivos:
    df_arq = pd.read_csv(arquivo, sep=',', encoding='utf-8-sig')
    
    # Renomear variações de observacao
    novas_colunas = []
    for c in df_arq.columns:
        if 'observacao' in c:
            novas_colunas.append('observacao')
        else:
            novas_colunas.append(c)
    df_arq.columns = novas_colunas
    
    # Consolidar colunas observacao duplicadas
    posicoes = [i for i, c in enumerate(df_arq.columns) if c == 'observacao']
    if len(posicoes) > 1:
        df_arq['observacao'] = df_arq.iloc[:, posicoes].apply(
            lambda row: ' | '.join([str(v).strip() for v in row if pd.notna(v) and str(v).strip() != '']),
            axis=1
        )
        df_arq = df_arq.loc[:, ~df_arq.columns.duplicated()]
    
    # Limpar tabs e espaços duplos
    df_arq['observacao'] = df_arq['observacao'].str.replace('\t', ' ', regex=False)
    df_arq['observacao'] = df_arq['observacao'].str.replace(r' +', ' ', regex=True)
    
    df_arq.to_csv(arquivo, index=False, encoding='utf-8-sig', sep=',')
    print(f"✓ {os.path.basename(arquivo)}")

print("\nPronto. Arquivos corrigidos na origem.")

✓ 2026_03_14_tribuna_liberal.csv
✓ 2026_03_18_tribuna_liberal.csv
✓ 2026_03_21_tribuna_liberal.csv
✓ 2026_03_24_tribuna_liberal.csv
✓ 2026_03_25_tribuna_liberal.csv
✓ 2026_03_26_tribuna_liberal.csv
✓ 2026_03_27_tribuna_liberal.csv
✓ 2026_03_28_tribuna_liberal.csv
✓ 2026_03_29_tribuna_liberal.csv
✓ 2026_03_31_tribuna_liberal.csv
✓ 2026_04_01_tribuna_liberal.csv

Pronto. Arquivos corrigidos na origem.


In [33]:
df = pd.concat(
    [pd.read_csv(f, sep=',', encoding='utf-8-sig') for f in arquivos],
    ignore_index=True
)

print(f"Total de registros: {len(df)}")
print(f"Total de colunas: {len(df.columns)}")
print(df.columns.tolist())

Total de registros: 38
Total de colunas: 16
['fonte', 'data', 'item', 'pagina', 'dimensao_ivs', 'codigo_variavel', 'tipo_relacao_variavel', 'resumo', 'localidade', 'tipo_evento', 'gravidade', 'observacao', 'cod_loteamento', 'nivel_confianca_loteamento', 'polaridade_evento', 'tipo_origem_evento']


In [2]:
import pandas as pd
import glob

# Lista todos os arquivos
arquivos = glob.glob("2026_*.csv")

dfs = []

for a in arquivos:
    try:
        df = pd.read_csv(
            a,
            sep=",",
            encoding="utf-8",
            engine="python",
            on_bad_lines="skip"
        )

        # limpeza básica
        df.columns = [str(c).strip() for c in df.columns]

        # garante colunas mínimas
        if "dimensao_ivs" not in df.columns:
            print(f"IGNORADO (sem estrutura): {a}")
            continue

        dfs.append(df)

    except Exception as e:
        print(f"ERRO em {a}: {e}")

# consolidação
df_total = pd.concat(dfs, ignore_index=True, sort=False)

# remove evento político (Clodoaldo)
df_total = df_total[
    ~df_total["item"].str.contains("Clodoaldo", na=False)
]

# remove registros sem polaridade
df_total = df_total[
    df_total["polaridade_evento"].notna()
]

# =========================
# RESULTADOS FINAIS
# =========================

print("=== BASE FINAL ===")
print("Total de eventos:", len(df_total))

print("\nPolaridade:")
print(df_total["polaridade_evento"].value_counts())

print("\nDimensões:")
print(df_total["dimensao_ivs"].value_counts())

ValueError: No objects to concatenate